简单RAG流程

In [55]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path=r"C:\Users\22349\RAG_Techniques\all_rag_techniques\untitled.env") #在这个文件夹下加入API_KEY
openai.api_key = os.getenv("OPENAI_API_KEY")
if api_key :
    print("成功加载 OPENAI_API_KEY:", api_key[:8] + "…")
else :
    print("没有加载到 OPENAI_API_KEY，请检查 .env 文件")

成功加载 OPENAI_API_KEY: sk-proj-…


In [56]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from pathlib import Path
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

llm=ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    max_tokens=1024
)
response = llm.invoke("Hi!")
print(response.content)
print('*'*50)
print(os.path.exists(".env"))#从 .env 文件中读取并加载环境变量。
load_dotenv(dotenv_path=r"C:\Users\22349\RAG_Techniques\all_rag_techniques\untitled.env")


Hello! 👋 How can I help you today?
**************************************************
False


True

# CSV文件的结构和使用案例

CSV 文件包含虚拟客户数据，包含各种属性，例如名字、姓氏、公司等。该数据集将用于 RAG 用例，以方便创建客户信息问答系统。

In [57]:
import os
import requests
# 1️. 创建 data 文件夹
os.makedirs('data', exist_ok=True)
print(os.listdir('.'))
# 2️.定义要下载的文件和对应 URL
files_to_download = {
    "data/Understanding_Climate_Change.pdf": "https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf",
    "data/customers-100.csv": "https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/customers-100.csv"
}
# 3️. 下载文件
for filepath, url in files_to_download.items():
    print(f"Downloading {url} → {filepath}")
    response = requests.get(url)
    response.raise_for_status()  # 出现错误会抛出异常
    with open(filepath, "wb") as f:
        f.write(response.content)

print("All files downloaded successfully!")

['.git', '.idea', 'all_rag_techniques', 'data', 'desktop.ini', 'RAG _base_study.ipynb', 'study_03.ipynb']
All files downloaded successfully!


In [69]:
import pandas as pd

file_path = ('data/customers-100.csv')#输入 CSV 文件的路径
data = pd.read_csv(file_path)
data.head(6)

,Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
0,1,DD37Cf93aecA6Dc,Sheryl,Baxter,Rasmussen Group,East Leonard,Chile,229.077.5154,397.884.0519x718,zunigavanessa@smith.info,2020-08-24,http://www.stephenson.com/
1,2,1Ef7b82A4CAAD10,Preston,Lozano,Vega-Gentry,East Jimmychester,Djibouti,5153435776,686-620-1820x944,vmata@colon.com,2021-04-23,http://www.hobbs.com/
2,3,6F94879bDAfE5a6,Roy,Berry,Murillo-Perry,Isabelborough,Antigua and Barbuda,+1-539-402-0259,(496)978-3969x58947,beckycarr@hogan.com,2020-03-25,http://www.lawrence.com/
3,4,5Cef8BFA16c5e3c,Linda,Olsen,"Dominguez, Mcmillan and Donovan",Bensonview,Dominican Republic,001-808-617-6467x12895,+1-813-324-8756,stanleyblackwell@benson.org,2020-06-02,http://www.good-lyons.com/
4,5,053d585Ab6b3159,Joanna,Bender,"Martin, Lang and Andrade",West Priscilla,Slovakia (Slovak Republic),001-234-203-0635x76146,001-199-446-3860x3486,colinalvarado@miles.net,2021-04-17,https://goodwin-ingram.com/
5,6,2d08FB17EE273F4,Aimee,Downs,Steele Group,Chavezborough,Bosnia and Herzegovina,(283)437-3886x88321,999-728-1637,louis27@gilbert.com,2020-02-25,http://www.berger.net/


加载和处理 CSV 数据

In [68]:
loader = CSVLoader(file_path=file_path)
docs = loader.load_and_split()

初始化 faiss 向量存储和 openai 嵌入

In [65]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=r"C:\Users\22349\RAG_Techniques\all_rag_techniques\untitled.env")

api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("成功加载 OPENAI_API_KEY:", api_key[:8] + "…")  # 只显示前8位，避免泄露
else:
    print("没有加载到 OPENAI_API_KEY，请检查 .env 文件")

成功加载 OPENAI_API_KEY: sk-proj-…


import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
index = faiss.IndexFlatL2(len(OpenAIEmbeddings().embed_query(" ")))
vector_store = FAISS(
    embedding_function=OpenAIEmbeddings(),
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)
将拆分后的 CSV 数据添加到向量存储库中

In [34]:
vector_store.add_documents(documents=docs)

NameError: name 'vector_store' is not defined


创建检索链

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

retriever = vector_store.as_retriever()

# Set up system prompt
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),

]）

# Create the question-answer chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)


根据 CSV 数据向 rag bot 提出问题

In [ ]:
answer= rag_chain.invoke({"input": "which company does sheryl Baxter work for?"})
answer['answer']